<a href="https://colab.research.google.com/github/Yisihaq01/breast-histology-calibration/blob/main/Notebook/04_external_data_preparation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from pathlib import Path
import shutil

EXTERNAL_ROOT = Path("/content/external_data")

BACH_ROOT = EXTERNAL_ROOT / "BACH"
BRACS_ROOT = EXTERNAL_ROOT / "BRACS"

for folder in [
    BACH_ROOT / "archives",
    BACH_ROOT / "images",
    BRACS_ROOT / "archives",
    BRACS_ROOT / "images",
    EXTERNAL_ROOT / "metadata",
]:
    folder.mkdir(parents=True, exist_ok=True)

disk = shutil.disk_usage("/content")

print("External data folder:", EXTERNAL_ROOT)
print(f"Free storage: {disk.free / 1024**3:.1f} GB")
print("\nBACH and BRACS folders are ready.")
print("Files in /content are temporary and need saving before runtime deletion.")

External data folder: /content/external_data
Free storage: 87.2 GB

BACH and BRACS folders are ready.
Files in /content are temporary and need saving before runtime deletion.


In [2]:
import hashlib
import requests
from tqdm.auto import tqdm

BACH_URL = (
    "https://zenodo.org/records/3632035/files/"
    "ICIAR2018_BACH_Challenge.zip?download=1"
)

BACH_ARCHIVE = (
    BACH_ROOT / "archives" / "ICIAR2018_BACH_Challenge.zip"
)

EXPECTED_MD5 = "8ae1801334aa943c44627c1eef3631b2"


def file_md5(path):
    digest = hashlib.md5()

    with path.open("rb") as file:
        for chunk in iter(lambda: file.read(8 * 1024 * 1024), b""):
            digest.update(chunk)

    return digest.hexdigest()


if BACH_ARCHIVE.exists():
    print("Archive already exists. Checking it before reuse.")

else:
    partial_path = BACH_ARCHIVE.with_suffix(".zip.part")

    with requests.get(
        BACH_URL,
        stream=True,
        timeout=(30, 180),
    ) as response:

        response.raise_for_status()
        total = int(response.headers.get("Content-Length", 0))

        with partial_path.open("wb") as file, tqdm(
            total=total or None,
            unit="B",
            unit_scale=True,
            desc="Downloading BACH",
        ) as progress:

            for chunk in response.iter_content(
                chunk_size=4 * 1024 * 1024
            ):
                if chunk:
                    file.write(chunk)
                    progress.update(len(chunk))

    actual_md5 = file_md5(partial_path)

    if actual_md5 != EXPECTED_MD5:
        raise RuntimeError(
            f"Checksum mismatch: {actual_md5}. "
            "The partial archive must not be used."
        )

    partial_path.rename(BACH_ARCHIVE)


actual_md5 = file_md5(BACH_ARCHIVE)

assert actual_md5 == EXPECTED_MD5, (
    f"Archive checksum mismatch: {actual_md5}"
)

print("\nPASS: BACH archive checksum verified.")
print("Archive:", BACH_ARCHIVE)
print(f"Size: {BACH_ARCHIVE.stat().st_size / 1024**3:.2f} GiB")


PASS: BACH archive checksum verified.
Archive: /content/external_data/BACH/archives/ICIAR2018_BACH_Challenge.zip
Size: 9.70 GiB


In [3]:
from collections import Counter
from pathlib import PurePosixPath
import zipfile

assert BACH_ARCHIVE.is_file()

with zipfile.ZipFile(BACH_ARCHIVE, "r") as archive:
    archive_files = [
        entry
        for entry in archive.infolist()
        if not entry.is_dir()
    ]

    print("Files in archive:", len(archive_files))

    folder_counts = Counter()
    image_counts = Counter()
    metadata_files = []

    image_extensions = {".tif", ".tiff", ".png", ".jpg", ".jpeg"}
    metadata_extensions = {
        ".csv", ".xlsx", ".xls", ".txt", ".md", ".json"
    }

    for entry in archive_files:
        path = PurePosixPath(entry.filename)

        # Ignore macOS bookkeeping files.
        if "__MACOSX" in path.parts or path.name.startswith("._"):
            continue

        folder = "/".join(path.parts[:2])
        folder_counts[folder] += 1

        if path.suffix.lower() in image_extensions:
            image_counts[str(path.parent)] += 1

        if path.suffix.lower() in metadata_extensions:
            metadata_files.append(entry.filename)

    print("\nTop-level folder counts:")
    for folder, count in sorted(folder_counts.items()):
        print(f"{folder}: {count}")

    print("\nImage counts by folder:")
    for folder, count in sorted(image_counts.items()):
        print(f"{folder}: {count}")

    print("\nPotential metadata/documentation files:")
    if metadata_files:
        for filename in sorted(metadata_files):
            print(filename)
    else:
        print("None found.")

    print("\nFirst 10 image paths:")
    image_paths = [
        entry.filename
        for entry in archive_files
        if PurePosixPath(entry.filename).suffix.lower()
        in image_extensions
        and "__MACOSX" not in PurePosixPath(entry.filename).parts
        and not PurePosixPath(entry.filename).name.startswith("._")
    ]

    for filename in sorted(image_paths)[:10]:
        print(filename)

print("\nPASS: Archive inventory completed. No files extracted.")

Files in archive: 488

Top-level folder counts:
ICIAR2018_BACH_Challenge/Photos: 407
ICIAR2018_BACH_Challenge/WSI: 81

Image counts by folder:
ICIAR2018_BACH_Challenge/Photos/Benign: 100
ICIAR2018_BACH_Challenge/Photos/InSitu: 100
ICIAR2018_BACH_Challenge/Photos/Invasive: 100
ICIAR2018_BACH_Challenge/Photos/Normal: 100
ICIAR2018_BACH_Challenge/WSI/gt_thumbnails: 10
ICIAR2018_BACH_Challenge/WSI/thumbnails: 30

Potential metadata/documentation files:
ICIAR2018_BACH_Challenge/Photos/microscopy_ground_truth.csv

First 10 image paths:
ICIAR2018_BACH_Challenge/Photos/Benign/b001.tif
ICIAR2018_BACH_Challenge/Photos/Benign/b002.tif
ICIAR2018_BACH_Challenge/Photos/Benign/b003.tif
ICIAR2018_BACH_Challenge/Photos/Benign/b004.tif
ICIAR2018_BACH_Challenge/Photos/Benign/b005.tif
ICIAR2018_BACH_Challenge/Photos/Benign/b006.tif
ICIAR2018_BACH_Challenge/Photos/Benign/b007.tif
ICIAR2018_BACH_Challenge/Photos/Benign/b008.tif
ICIAR2018_BACH_Challenge/Photos/Benign/b009.tif
ICIAR2018_BACH_Challenge/Photos/